---
## Module 5 - Generative AI (Innermost Ring)

**Generative AI** can produce new content (text, images, code, audio) indistinguishable from human-created artifacts.

| Topic | Description |
|-------|-------------|
| Language Modeling | Predicting the next token in a sequence |
| Transformer Architecture | Self-attention-based seq2seq architecture |
| Self-Attention | Each token attends to all others |
| Natural Language Understanding | Classification, NER, intent detection |
| Text Generation | Autoregressive decoding from LLMs |
| Summarization | Condensing long text to key points |
| Dialogue Systems | Multi-turn conversational AI |


### 5.1 Language Modeling - N-Gram Model

In [ ]:
import re
from collections import defaultdict, Counter

subprocess.run(['wget', '-q', '-O', '/content/alice.txt',
    'https://www.gutenberg.org/files/11/11-0.txt'], check=True)

with open('/content/alice.txt', 'r', encoding='utf-8', errors='ignore') as f:
    raw = f.read()[5000:35000]

tokens = re.findall(r"[a-z']+", raw.lower())
print(f"Corpus tokens: {len(tokens):,}")

class NGramLM:
    def __init__(self, n=3):
        self.n = n
        self.model = defaultdict(Counter)
    def train(self, tokens):
        for i in range(len(tokens)-self.n):
            ctx   = tuple(tokens[i:i+self.n-1])
            next_ = tokens[i+self.n-1]
            self.model[ctx][next_] += 1
    def predict_next(self, context, top_k=5):
        ctx = tuple(context[-(self.n-1):])
        if ctx not in self.model: return []
        total = sum(self.model[ctx].values())
        return [(w, c/total) for w,c in self.model[ctx].most_common(top_k)]
    def generate(self, seed, n_words=30):
        result = list(seed)
        for _ in range(n_words):
            preds = self.predict_next(result)
            if not preds: break
            words, probs = zip(*preds)
            probs = np.array(probs, dtype=np.float64)
            probs = probs / probs.sum()          #  re-normalize to fix float drift
            result.append(np.random.choice(words, p=probs))
        return ' '.join(result)

lm3 = NGramLM(n=3); lm3.train(tokens)
lm4 = NGramLM(n=4); lm4.train(tokens)

seed = ['alice', 'was']
gen3 = lm3.generate(seed, 30)
gen4 = lm4.generate(seed, 30)
print(f"\n3-gram generation:\n  {gen3}")
print(f"\n4-gram generation:\n  {gen4}")

freq = Counter(tokens)
top20 = freq.most_common(20)
words20, counts20 = zip(*top20)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].barh(list(words20)[::-1], list(counts20)[::-1],
             color=plt.cm.Blues_r(np.linspace(0.3,0.9,20)))
axes[0].set_title('Top 20 Tokens  Alice in Wonderland', fontweight='bold')
axes[0].set_xlabel('Count')

preds = lm3.predict_next(seed, top_k=8)
if preds:
    ws, ps = zip(*preds)
    axes[1].bar(ws, ps, color=plt.cm.Greens(np.linspace(0.4,0.9,len(ws))))
    axes[1].set_title(f'Next-word Probabilities\nContext: "{" ".join(seed)}"', fontweight='bold')
    axes[1].set_xlabel('Next Word'); axes[1].set_ylabel('Probability')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/05_ngram_lm.png', dpi=130, bbox_inches='tight')
plt.show()

### 5.2 Transformer Architecture - Build from Scratch

In [ ]:
#  Minimal Transformer encoder in Keras/TF 
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        assert embed_dim % num_heads == 0
        self.depth = embed_dim // num_heads
        self.Wq = layers.Dense(embed_dim)
        self.Wk = layers.Dense(embed_dim)
        self.Wv = layers.Dense(embed_dim)
        self.dense = layers.Dense(embed_dim)
    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0,2,1,3])
    def call(self, x):
        bs = tf.shape(x)[0]
        Q  = self.split_heads(self.Wq(x), bs)
        K  = self.split_heads(self.Wk(x), bs)
        V  = self.split_heads(self.Wv(x), bs)
        scale  = tf.cast(self.depth, tf.float32) ** -0.5
        attn   = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) * scale, axis=-1)
        out    = tf.matmul(attn, V)
        out    = tf.transpose(out, [0,2,1,3])
        out    = tf.reshape(out, (bs, -1, self.embed_dim))
        return self.dense(out)

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ff    = keras.Sequential([layers.Dense(ff_dim,'relu'), layers.Dense(embed_dim)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)
    def call(self, x, training=False):
        attn_out = self.drop1(self.attn(x), training=training)
        x = self.norm1(x + attn_out)
        ff_out = self.drop2(self.ff(x), training=training)
        return self.norm2(x + ff_out)

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb   = layers.Embedding(maxlen, embed_dim)
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

#  Train for sentiment classification (IMDb) 
VOCAB_SIZE, MAXLEN, EMBED_DIM, NUM_HEADS, FF_DIM = 10000, 200, 64, 4, 128
(X_im_tr, y_im_tr), (X_im_te, y_im_te) = imdb.load_data(num_words=VOCAB_SIZE)
X_im_tr = keras.preprocessing.sequence.pad_sequences(X_im_tr, maxlen=MAXLEN)
X_im_te = keras.preprocessing.sequence.pad_sequences(X_im_te, maxlen=MAXLEN)

inp = layers.Input(shape=(MAXLEN,))
x = TokenAndPositionEmbedding(MAXLEN, VOCAB_SIZE, EMBED_DIM)(inp)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation='relu')(x)
x = layers.Dropout(0.1)(x)
out = layers.Dense(1, activation='sigmoid')(x)
transformer_model = keras.Model(inp, out, name='Transformer_IMDb')
transformer_model.summary()

transformer_model.compile('adam', 'binary_crossentropy', metrics=['accuracy'])
h_tf = transformer_model.fit(X_im_tr, y_im_tr, epochs=8, batch_size=64,
                               validation_split=0.1, verbose=1)
_, tf_acc = transformer_model.evaluate(X_im_te, y_im_te, verbose=0)
print(f"\n Transformer IMDb Sentiment Accuracy: {tf_acc*100:.2f}%")

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(h_tf.history['accuracy'],     lw=2, label='Train')
axes[0].plot(h_tf.history['val_accuracy'], lw=2, label='Val')
axes[0].set_title('Transformer - IMDb Accuracy', fontweight='bold'); axes[0].legend()
axes[1].plot(h_tf.history['loss'],     lw=2, color='orange', label='Train')
axes[1].plot(h_tf.history['val_loss'], lw=2, color='red', label='Val')
axes[1].set_title('Transformer - IMDb Loss', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/05_transformer.png', dpi=130, bbox_inches='tight'); plt.show()

### 5.3 Pretrained Transformers - HuggingFace: NLU, Generation & Summarization

In [ ]:
from transformers import pipeline, AutoTokenizer

#  1. Sentiment Analysis (NLU) 
print("=== 1. Sentiment Analysis ===")
sentiment = pipeline("sentiment-analysis",
                     model="distilbert-base-uncased-finetuned-sst-2-english")
texts_sent = [
    "Artificial Intelligence is transforming every industry.",
    "This neural network converges too slowly - frustrating!",
    "The transformer architecture is an elegant and powerful design.",
    "Overfitting ruined my model completely.",
]
sent_results = sentiment(texts_sent)
df_sent = pd.DataFrame([{'Text': t[:55]+'...', 'Label': r['label'],
                          'Score': f"{r['score']:.3f}"}
                         for t,r in zip(texts_sent, sent_results)])
print(df_sent.to_string(index=False))

#  2. Zero-shot Classification 
print("\n=== 2. Zero-Shot Classification ===")
zsc = pipeline("zero-shot-classification",
               model="facebook/bart-large-mnli")
text_zsc = "This paper proposes a novel quantum-resistant encryption algorithm for IoT devices."
labels   = ["cybersecurity","machine learning","quantum computing","robotics","healthcare"]
res_zsc  = zsc(text_zsc, candidate_labels=labels)
df_zsc   = pd.DataFrame({'Label': res_zsc['labels'],
                          'Score': [f'{s:.3f}' for s in res_zsc['scores']]})
print(f"Input: '{text_zsc[:60]}...'")
print(df_zsc.to_string(index=False))

#  3. Summarization (direct model call  pipeline task removed in this transformers version) 
print("\n=== 3. Summarization ===")
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

sum_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_model     = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

long_text = (
    "Deep learning has revolutionized AI by enabling machines to learn hierarchical "
    "representations from raw data. Unlike traditional ML that requires manual feature "
    "engineering, deep learning models automatically extract features at multiple levels. "
    "This has led to breakthroughs in computer vision, NLP, speech recognition, and drug "
    "discovery. The transformer architecture replaced recurrence with self-attention, enabling "
    "parallel training on massive datasets and giving rise to LLMs like GPT and BERT."
)

inputs     = sum_tokenizer(long_text, return_tensors="pt", max_length=512, truncation=True)
sum_ids    = sum_model.generate(inputs["input_ids"], max_length=60, min_length=20,
                                 length_penalty=2.0, num_beams=4, early_stopping=True)
summary    = sum_tokenizer.decode(sum_ids[0], skip_special_tokens=True)

print(f"Original ({len(long_text.split())} words):\n{long_text[:200]}...")
print(f"\nSummary  ({len(summary.split())} words):\n{summary}")

#  4. Text Generation (GPT-2)  pipeline('text-generation') is still supported 
print("\n=== 4. Text Generation (GPT-2) ===")
generator_hf = pipeline("text-generation", model="gpt2", max_new_tokens=60)
prompt = "The future of artificial intelligence in education is"
gen_text = generator_hf(prompt, do_sample=True, temperature=0.8, num_return_sequences=2)
for i, g in enumerate(gen_text):
    print(f"\nGeneration {i+1}:\n  {g['generated_text']}")

### 5.4 Dialogue Systems - Retrieval-Based Chatbot

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#  Build a retrieval-based QA chatbot 
knowledge_base = {
    "What is a neural network?":
        "A neural network is a computational model inspired by the human brain, composed of layers of interconnected nodes (neurons) that learn patterns from data via backpropagation.",
    "What is a transformer?":
        "A transformer is a deep learning architecture based on self-attention mechanisms. It processes entire sequences in parallel and is the foundation for large language models like BERT and GPT.",
    "What is deep learning?":
        "Deep learning is a subset of machine learning using multi-layered neural networks to automatically learn hierarchical feature representations from raw data.",
    "What is generative AI?":
        "Generative AI refers to AI systems that can create new content (text, images, code, music) by learning the statistical distribution of training data.",
    "What is backpropagation?":
        "Backpropagation is the algorithm for training neural networks. It computes gradients of the loss with respect to each weight using the chain rule, then updates weights via gradient descent.",
    "What is overfitting?":
        "Overfitting occurs when a model learns the training data too well, including noise, causing poor generalization to unseen data. It is mitigated by dropout, regularization, and more data.",
    "What is an attention mechanism?":
        "An attention mechanism allows a model to selectively focus on different parts of the input when producing each output element, enabling capture of long-range dependencies.",
    "What is reinforcement learning?":
        "Reinforcement learning is a paradigm where an agent learns to take actions in an environment to maximize cumulative reward through trial and error.",
}

questions = list(knowledge_base.keys())
answers   = list(knowledge_base.values())

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(questions)

def chat(user_input, threshold=0.25):
    vec = vectorizer.transform([user_input])
    sims = cosine_similarity(vec, tfidf_matrix)[0]
    best_idx  = np.argmax(sims)
    best_score = sims[best_idx]
    if best_score < threshold:
        return "I don't have information on that topic yet.", best_score
    return answers[best_idx], best_score

# Demo conversation
test_queries = [
    "Can you explain what neural networks are?",
    "How does the transformer work?",
    "Tell me about overfitting problems",
    "What is the attention mechanism?",
    "How do we train models?",
    "What can generative AI do?",
]
rows_chat = []
for q in test_queries:
    response, score = chat(q)
    rows_chat.append({'User Query': q, 'Confidence': f'{score:.3f}',
                       'Bot Response': response[:80]+'...'})
df_chat = pd.DataFrame(rows_chat)
print("=== Retrieval-Based Chatbot Demo ===")
print(df_chat.to_string(index=False))

# Similarity heatmap
all_inputs = test_queries + questions
all_vecs   = vectorizer.transform(all_inputs)
sim_matrix = cosine_similarity(all_vecs[:len(test_queries)], all_vecs[len(test_queries):])
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[q[:35]+'...' for q in questions],
            yticklabels=[q[:35]+'...' for q in test_queries], ax=ax)
ax.set_title('Chatbot TF-IDF Similarity - Query vs Knowledge Base', fontweight='bold')
ax.set_xlabel('Knowledge Base Questions'); ax.set_ylabel('User Queries')
plt.xticks(rotation=35, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/05_chatbot_similarity.png', dpi=130, bbox_inches='tight'); plt.show()

---
## Module 6 -- Frontier Generative AI (Chapters 26-29)

The frontier of modern AI: diffusion models, LoRA fine-tuning, RLHF alignment, and Mamba SSMs.

| Section | Chapter | Topic |
|---------|---------|-------|
| 6.1 | Ch 26 | Diffusion Models (DDPM on MNIST) |
| 6.2 | Ch 27 | LoRA / PEFT fine-tuning (GPT-2) |
| 6.3 | Ch 28 | RLHF alignment pipeline (PPO) |
| 6.4 | Ch 29 | Mamba / State Space Models |

### 6.1 Diffusion Models -- DDPM on MNIST (Chapter 26)

A **Denoising Diffusion Probabilistic Model (DDPM)** learns to reverse a noise-addition process.
Forward: $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon$. Reverse: a network predicts the noise and denoises step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

T = 200; BATCH = 128; EPOCHS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
betas = torch.linspace(1e-4, 0.02, T).to(DEVICE)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise=None):
    if noise is None: noise = torch.randn_like(x0)
    ab = alpha_bar[t].view(-1,1,1,1)
    return torch.sqrt(ab)*x0 + torch.sqrt(1-ab)*noise, noise

class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),
                                  nn.Conv2d(32,64,3,padding=1,stride=2),nn.ReLU())
        self.mid = nn.Sequential(nn.Conv2d(64,64,3,padding=1),nn.ReLU())
        self.dec = nn.Sequential(nn.ConvTranspose2d(64,32,2,stride=2),nn.ReLU(),
                                  nn.Conv2d(32,1,3,padding=1))
        self.t_emb = nn.Embedding(T, 64)
    def forward(self, x, t):
        te = self.t_emb(t).view(-1,64,1,1)
        e  = self.enc(x)
        m  = self.mid(e + te)
        return self.dec(m)

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,),(0.5,))])
loader = DataLoader(datasets.MNIST('/tmp',download=True,transform=tf),
                    batch_size=BATCH, shuffle=True)
model = SimpleUNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=2e-4)
losses = []
print(f'Training DDPM on {DEVICE} for {EPOCHS} epochs...')
for epoch in range(EPOCHS):
    epoch_loss = 0
    for x, _ in loader:
        x = x.to(DEVICE)
        t = torch.randint(0, T, (x.shape[0],), device=DEVICE)
        xt, noise = q_sample(x, t)
        pred = model(xt, t)
        loss = nn.functional.mse_loss(pred, noise)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    print(f'  Epoch {epoch+1}/{EPOCHS}  loss={avg:.4f}')

@torch.no_grad()
def p_sample_loop(model, shape):
    x = torch.randn(shape, device=DEVICE)
    for t_val in reversed(range(T)):
        tb = torch.full((shape[0],), t_val, device=DEVICE, dtype=torch.long)
        b = betas[t_val]; a = alphas[t_val]; ab = alpha_bar[t_val]
        eps = model(x, tb)
        x = (x - (b/torch.sqrt(1-ab))*eps) / torch.sqrt(a)
        if t_val > 0: x += torch.sqrt(b)*torch.randn_like(x)
    return x.clamp(-1,1)

samples = p_sample_loop(model, (16,1,28,28))
fig, axes = plt.subplots(2, 8, figsize=(14,4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i,0].cpu().numpy(), cmap='gray'); ax.axis('off')
plt.suptitle(f'DDPM Generated Digits (after {EPOCHS} epochs)', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI_Universe_Course/07_ddpm_samples.png', dpi=130)
plt.show()
print('Listing 26.1 complete')

### 6.2 LoRA / PEFT -- Parameter-Efficient Fine-Tuning (Chapter 27)

**LoRA** freezes pretrained weights and adds trainable low-rank matrices $\Delta W = BA$ with rank $r \ll \min(d,k)$. Fine-tunes under 1% of parameters while matching full fine-tuning quality.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','torchao>=0.16.0','peft','transformers','datasets'])

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

MODEL_NAME = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

texts = [
    'Quantum computing uses superposition and entanglement.',
    'A qubit can represent 0 and 1 simultaneously.',
    'Grovers algorithm searches databases in O(sqrt(N)) time.',
    'VQE estimates molecular ground state energies on quantum hardware.',
    'QAOA solves combinatorial optimisation using quantum circuits.',
    'Diffusion models learn to reverse a noise-addition process.',
    'LoRA adds low-rank adapter matrices to frozen model weights.',
    'RLHF aligns language models to human preferences via reward modelling.',
    'Mamba processes sequences in linear time using state spaces.',
    'Transformers use self-attention for long-range dependencies.',
] * 6

def tokenize(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=64,
                    padding='max_length', return_tensors='pt')
    enc['labels'] = enc['input_ids'].clone()
    return {k: v.squeeze(0) for k,v in enc.items()}

ds = Dataset.from_dict({'text': texts}).map(tokenize, remove_columns=['text'])

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total_params = sum(p.numel() for p in model.parameters())

lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32,
                       target_modules=['c_attn'], lora_dropout=0.05, bias='none')
model = get_peft_model(model, lora_cfg)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable (LoRA r=8): {trainable:,}  ({100*trainable/total_params:.2f}%)')

args = TrainingArguments(output_dir='/tmp/lora_gpt2', num_train_epochs=3,
    per_device_train_batch_size=4, learning_rate=3e-4,
    logging_steps=10, save_strategy='no', report_to='none',
    fp16=torch.cuda.is_available())
Trainer(model=model, args=args, train_dataset=ds).train()

model.eval()
inputs = tokenizer('Quantum computing', return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.7,
                      pad_token_id=tokenizer.eos_token_id)
print('Generated:', tokenizer.decode(out[0], skip_special_tokens=True))
print('Listing 27.1 complete')

### 6.3 RLHF -- Reinforcement Learning from Human Feedback (Chapter 28)

RLHF transforms a base LM into an instruction-following assistant through:
(1) Supervised Fine-Tuning, (2) Reward Model training on preference pairs,
(3) PPO policy optimisation with KL penalty to prevent reward hacking.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torch.optim import Adam
import copy

def reward_fn(responses):
    return [-abs(len(r.split()) - 30) / 30.0 for r in responses]

MODEL_NAME = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

lora_cfg = LoraConfig(r=4, lora_alpha=16, target_modules=['c_attn'],
                       lora_dropout=0.05, bias='none')
policy = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
policy = get_peft_model(policy, lora_cfg)
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
ref_model.eval()
for p in ref_model.parameters(): p.requires_grad_(False)

optimizer = Adam(policy.parameters(), lr=1.41e-5)
KL_COEF = 0.2

prompts = ['Explain a neural network.','What is machine learning?',
           'Describe transformers.','What is reinforcement learning?',
           'Explain gradient descent.','What are activation functions?',
           'Describe backpropagation.','What is transfer learning?']

rewards_log, kl_log = [], []
print('Running 5 PPO steps...')
for step in range(5):
    policy.eval()
    pt = [tokenizer(p, return_tensors='pt').input_ids.squeeze() for p in prompts]
    with torch.no_grad():
        rt = [policy.generate(t.unsqueeze(0), max_new_tokens=50, do_sample=True,
              temperature=0.7, pad_token_id=tokenizer.eos_token_id).squeeze()
              for t in pt]
    responses = [tokenizer.decode(r, skip_special_tokens=True) for r in rt]
    rewards   = torch.tensor(reward_fn(responses))

    # KL penalty: log policy(response) - log ref(response)
    policy.train()
    total_loss = torch.tensor(0.)
    kls = []
    for i, (inp, out) in enumerate(zip(pt, rt)):
        ids = out.unsqueeze(0)
        with torch.no_grad():
            ref_logits = ref_model(ids).logits
        pol_logits = policy(ids).logits
        log_ratio = (pol_logits - ref_logits).mean()
        kls.append(log_ratio.item())
        advantage = rewards[i] - KL_COEF * log_ratio.detach()
        loss = -advantage * pol_logits.mean()
        total_loss = total_loss + loss

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    mean_r = rewards.mean().item()
    mean_kl = np.mean(kls)
    rewards_log.append(mean_r); kl_log.append(mean_kl)
    print(f'  Step {step+1}: reward={mean_r:.3f}  KL={mean_kl:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,3))
ax1.plot(rewards_log,'g-o',ms=6); ax1.set_title('Mean Reward'); ax1.set_xlabel('PPO Step')
ax2.plot(kl_log,'r-o',ms=6); ax2.set_title('KL Divergence'); ax2.set_xlabel('PPO Step')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI_Universe_Course/07_rlhf_training.png', dpi=130)
plt.show()
print('Listing 28.1 complete')

### 6.4 Mamba / State Space Models (Chapter 29)

**Mamba** processes sequences in $O(n)$ time via selective SSM recurrence:
$h_t = A_t h_{t-1} + B_t x_t$, $y_t = C_t h_t$.
Unlike Transformers (quadratic attention), the hidden state size is constant
regardless of sequence length. Selectivity makes $B_t, C_t$ input-dependent.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt

class SelectiveSSM(nn.Module):
    def __init__(self, d_model=32, d_state=16):
        super().__init__()
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, d_model)
        self.A_log   = nn.Parameter(torch.randn(d_model, d_state))
        self.B_proj  = nn.Linear(d_model, d_state)
        self.C_proj  = nn.Linear(d_model, d_state)
        self.D       = nn.Parameter(torch.ones(d_model))
        self.out_proj= nn.Linear(d_model, d_model)
    def forward(self, x):
        B, L, D = x.shape
        u  = self.in_proj(x)
        A  = -torch.exp(self.A_log)
        Bs = self.B_proj(x)
        Cs = self.C_proj(x)
        h  = torch.zeros(B, D, self.d_state, device=x.device)
        ys = []
        for t in range(L):
            dA = torch.exp(A.unsqueeze(0))
            dB = Bs[:,t,:].unsqueeze(1) * u[:,t,:].unsqueeze(2)
            h  = dA * h + dB
            y  = (h * Cs[:,t,:].unsqueeze(1)).sum(-1)
            ys.append(y)
        ys = torch.stack(ys, dim=1)
        ys = ys + self.D.unsqueeze(0).unsqueeze(0) * u
        return self.out_proj(ys)

class MambaModel(nn.Module):
    def __init__(self, vocab=3, d_model=32, d_state=16):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.ssm   = SelectiveSSM(d_model, d_state)
        self.head  = nn.Linear(d_model, vocab)
    def forward(self, x): return self.head(self.ssm(self.embed(x)))

SEQ_LEN = 64; GAP = SEQ_LEN // 2; VOCAB = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def make_batch(batch_size=64):
    seqs = torch.zeros(batch_size, SEQ_LEN, dtype=torch.long)
    tgts = torch.zeros(batch_size, SEQ_LEN, dtype=torch.long)
    tokens = torch.randint(0, VOCAB, (batch_size, GAP))
    seqs[:, :GAP] = tokens
    tgts[:, GAP:GAP*2] = tokens
    return seqs.to(DEVICE), tgts.to(DEVICE)

model = MambaModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
losses = []
print(f'Training selective SSM on {SEQ_LEN}-step copying task...')
for step in range(300):
    x, y = make_batch()
    loss = criterion(model(x).reshape(-1, VOCAB), y.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 50 == 0: print(f'  Step {step:3d}: loss={loss.item():.4f}')
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    x, y = make_batch(256)
    preds = model(x).argmax(-1)
    acc = (preds[:, GAP:GAP*2] == y[:, GAP:GAP*2]).float().mean().item()
    print(f'Copy accuracy: {acc*100:.1f}%')

plt.figure(figsize=(8,3))
plt.plot(losses, alpha=0.7)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title(f'Mamba SSM -- {SEQ_LEN}-step Copy Task (acc={acc*100:.1f}%)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI_Universe_Coding_Guide/07_mamba_loss.png', dpi=130)
plt.show()
print('Listing 29.1 complete')

---
## Stretch Goals -- Chapters 2-29

One hands-on coding extension per chapter, going beyond the base listing.
Run these after completing the corresponding module section.

| Cell | Chapter | Stretch Goal Summary |
|------|---------|----------------------|
| SG-02 | Ch 2  | A* diagonal heuristic + weighted terrain |
| SG-03 | Ch 3  | Expert system + confidence scores |
| SG-04 | Ch 4  | Fuzzy system + third input variable |
| SG-05 | Ch 5  | Polynomial features + correlation filter pipeline |
| SG-06 | Ch 6  | +2 classifiers + stratified cross-validation |
| SG-07 | Ch 7  | DBSCAN + hierarchical clustering + dendrogram |
| SG-08 | Ch 8  | Self-training SSL loop |
| SG-09 | Ch 9  | Stacking ensemble with meta-learner |
| SG-10 | Ch 10 | GELU from scratch + MLP comparison |
| SG-11 | Ch 11 | Pocket Algorithm on noisy XOR |
| SG-12 | Ch 12 | 3-layer backprop + L2 weight decay |
| SG-13 | Ch 13 | TCN vs LSTM on sine wave |
| SG-14 | Ch 14 | SOM on Digits 64-feature dataset |
| SG-15 | Ch 15 | CNN depth study vs MLP on CIFAR-10 |
| SG-16 | Ch 16 | Unfreeze top 20 layers fine-tuning |
| SG-17 | Ch 17 | Conditional GAN (cGAN) on MNIST |
| SG-18 | Ch 18 | Causal masked self-attention |
| SG-19 | Ch 19 | MC Dropout uncertainty estimation |
| SG-20 | Ch 20 | Double DQN vs vanilla Q-Learning |
| SG-21 | Ch 21 | Routing-by-agreement coupling coefficient heatmap |
| SG-22 | Ch 22 | Add-k smoothing + linear interpolation |
| SG-23 | Ch 23 | 2-block Transformer + attention visualisation |
| SG-24 | Ch 24 | DistilBERT fine-tune vs zero-shot BART |
| SG-25 | Ch 25 | Dense retrieval RAG with sentence-transformers |
| SG-26 | Ch 26 | Cosine vs linear noise schedule DDPM |
| SG-27 | Ch 27 | LoRA rank sweep r=1,4,8,16 |
| SG-28 | Ch 28 | RLHF reward hacking with length reward |
| SG-29 | Ch 29 | Mamba vs LSTM at seq lengths 64,128,256 |